<div style="background: linear-gradient(135deg, #1F3864 0%, #2E5FA3 100%); padding: 48px 40px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: white; font-size: 2.4em; font-weight: 800; margin: 0 0 8px 0; letter-spacing: -0.5px;">Deep Learning for Business Analytics</h1>
  <h2 style="color: #a8c4e8; font-size: 1.3em; font-weight: 400; margin: 0 0 16px 0; font-style: italic;">From Basics to Large Language Models</h2>
  <p style="color: #c8ddf5; font-size: 0.95em; margin: 0 0 24px 0;">Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter</p>
  <div style="background: rgba(255,255,255,0.12); border-radius: 8px; padding: 16px 20px; display: inline-block;">
    <span style="color: #ddeeff; font-size: 1.05em; font-weight: 600;">&#128214; Chapter 5 &nbsp;&middot;&nbsp; Convolutional Neural Networks</span>
  </div>
</div>
<div style="background: #f0f4fa; border-left: 5px solid #2E5FA3; padding: 14px 20px; border-radius: 0 8px 8px 0; margin-top: 4px; color: #333; font-size: 0.97em;">
  <em>How CNNs learn to see &mdash; from convolution and pooling to detecting manufacturing defects in product images.</em>
</div>

## What This Chapter Covers

| Section | Topics |
|---------|--------|
| 5.1 How Convolution Works | Why FFNs fail on images · Kernels and feature maps · Stride and padding |
| 5.2 Pooling and CNN Architecture | Max pooling · Conv→ReLU→Pool blocks · Fully connected head |
| 5.3 Training and Evaluation | Data augmentation · Visualising learned filters · Comparing CNN vs FFN |
| 5.4 Business Application: Defect Detection | Binary CNN · Precision/recall trade-off · Threshold tuning |
| 5.5 Pipeline: Saving and Deploying the Defect Detector | `ModelPipeline` with image transforms · Shape/channel validation · Retrain |

> **Datasets used in this chapter:**
> - **Fashion-MNIST** — loaded via `torchvision.datasets` (no download required)
> - **Kaggle Casting Product Defect Images** — downloaded in Section 5.4

---
## Setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Chapter 5 — Setup
# Run this cell first. Expected time: under 2 minutes on Colab.
# ─────────────────────────────────────────────────────────────────────────────

!pip install --quiet torch torchvision numpy pandas matplotlib seaborn scikit-learn kaggle tqdm dill

import copy, datetime, os, json, getpass
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

np.random.seed(42)
torch.manual_seed(42)

device = (
    torch.device('cuda') if torch.cuda.is_available() else
    torch.device('mps')  if hasattr(torch.backends, 'mps') and
                             torch.backends.mps.is_available() else
    torch.device('cpu')
)

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#f8f9fa'
plt.rcParams['axes.grid']        = True
plt.rcParams['grid.alpha']       = 0.4
plt.rcParams['font.size']        = 11

print(f'Setup complete. Running on: {device}')

---
## Reusable Training Helpers

The same three training functions and `EarlyStopping` class from Chapters 3 and 4 are used here unchanged. They are defined once at the top and called for both Fashion-MNIST and the defect detector.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Training helpers — identical to Chapters 3 and 4
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimiser, device):
    """One full pass over the training data. Returns mean loss."""
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimiser.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimiser.step()
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    """Evaluate without updating weights. Returns mean loss."""
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            total_loss += criterion(model(X_batch), y_batch).item() * len(X_batch)
    return total_loss / len(loader.dataset)


def run_training(model, train_loader, val_loader, criterion, optimiser,
                 num_epochs, device, scheduler=None, val_loss_scheduler=None,
                 early_stopping=None, verbose_every=10):
    """Train for num_epochs. Returns (train_losses, val_losses).

    scheduler          : epoch-based scheduler (StepLR, CosineAnnealingLR)
    val_loss_scheduler : loss-based scheduler (ReduceLROnPlateau)
    early_stopping     : EarlyStopping instance
    """
    train_losses, val_losses = [], []
    for epoch in range(num_epochs):
        tl = train_one_epoch(model, train_loader, criterion, optimiser, device)
        vl = evaluate(model, val_loader, criterion, device)
        train_losses.append(tl)
        val_losses.append(vl)
        if scheduler:
            scheduler.step()
        if val_loss_scheduler:
            val_loss_scheduler.step(vl)
        if verbose_every and (epoch % verbose_every == 0 or epoch == num_epochs - 1):
            print(f'  Epoch {epoch:3d}: train={tl:.4f}  val={vl:.4f}')
        if early_stopping and early_stopping.step(vl, model):
            print(f'  Early stopping at epoch {epoch}  (best val: {early_stopping.best_loss:.4f})')
            early_stopping.restore_best(model)
            break
    return train_losses, val_losses


class EarlyStopping:
    """Halt training when validation loss stops improving. Identical to Chapters 3 and 4."""
    def __init__(self, patience=8, min_delta=1e-4):
        self.patience     = patience
        self.min_delta    = min_delta
        self.best_loss    = float('inf')
        self.counter      = 0
        self.best_weights = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss    = val_loss
            self.counter      = 0
            self.best_weights = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model):
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)


print('Helpers defined: train_one_epoch | evaluate | run_training | EarlyStopping')

---
# 5.1 How Convolution Works

## Why FFNs struggle with images

In Chapter 4 we flattened each 28×28 MNIST image into a vector of 784 numbers and fed it straight into a fully connected layer. This works for small, centred images like MNIST, but it has two fundamental problems when applied to real image data.

**Problem 1 — Scale.** A modest 224×224 colour image has 224 × 224 × 3 = 150,528 pixel values. A single fully connected layer mapping this to 512 neurons would have over 77 million parameters — just in that one layer. Training becomes impractically slow, and the model is very likely to overfit.

**Problem 2 — No spatial awareness.** When you flatten an image, you destroy its structure. Pixel (row=10, col=15) and the pixel next to it (row=10, col=16) become unrelated numbers 15 positions apart in a vector. An FFN has no way to learn that neighbouring pixels are related — yet spatial proximity is exactly what makes images recognisable.

## The convolutional solution

A **convolutional layer** solves both problems by replacing the fully connected weight matrix with a small **kernel** (also called a filter) — typically 3×3 or 5×5 pixels — that slides across the image. At each position, it computes the dot product between the kernel weights and the patch of image pixels underneath it. This produces one output value. Sliding across the entire image produces a **feature map**.

This approach has two key properties:
- **Parameter sharing** — the same kernel weights are reused at every position. A 3×3 kernel has only 9 weights regardless of how large the image is, solving the scale problem.
- **Local connectivity** — each output value only depends on a small neighbourhood of input pixels, preserving spatial structure.

A single convolutional layer typically learns many kernels in parallel (e.g. 32 or 64), each detecting a different local pattern — one might activate on horizontal edges, another on diagonal lines, another on colour transitions.

## What a kernel is — step by step

A **kernel** is a small grid of numbers — typically 3×3 — that acts like a magnifying glass sliding across the image. At each position, it performs a **dot product**: it multiplies each kernel value by the image pixel underneath it, then adds all the products together into one number. That number is written into the output (the **feature map**).

The diagram below shows a 3×3 kernel applied to one position of a 5×5 image patch:

```
Image patch (3×3)       Kernel (3×3)          Output value
┌───┬───┬───┐           ┌────┬────┬────┐
│ 1 │ 0 │ 1 │           │ -1 │ -2 │ -1 │
├───┼───┼───┤     ×     ├────┼────┼────┤   =   1×(-1) + 0×(-2) + 1×(-1)
│ 0 │ 1 │ 0 │           │  0 │  0 │  0 │       + 0×0 + 1×0 + 0×0
├───┼───┼───┤           ├────┼────┼────┤       + 1×1 + 0×2 + 1×1  =  0
│ 1 │ 0 │ 1 │           │  1 │  2 │  1 │
└───┴───┴───┘           └────┴────┴────┘
```

The kernel then **slides to the next position** and repeats. Sliding across every position of the image produces the full feature map. The kernel weights are the same at every position — this is **parameter sharing**.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 5.0a — How a kernel slides across an image (stride and padding)
# ─────────────────────────────────────────────────────────────────────────────

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

def draw_grid(ax, rows, cols, x0, y0, cell, vals=None,
              facecolor='white', edgecolor='#555', fontsize=9, bold=False):
    for r in range(rows):
        for c in range(cols):
            rect = plt.Rectangle((x0 + c*cell, y0 - r*cell), cell, cell,
                                  facecolor=facecolor, edgecolor=edgecolor, lw=1.0)
            ax.add_patch(rect)
            if vals is not None:
                v = vals[r][c]
                ax.text(x0 + c*cell + cell/2, y0 - r*cell + cell/2,
                        str(v), ha='center', va='center',
                        fontsize=fontsize,
                        fontweight='bold' if bold else 'normal',
                        color='#1a1a1a')

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
fig.patch.set_facecolor('white')
titles = [
    'Stride = 1, no padding\nKernel moves 1 pixel at a time',
    'Stride = 2, no padding\nKernel jumps 2 pixels — output is smaller',
    'Stride = 1, padding = 1\nZero border added — output same size as input',
]

for ax_idx, ax in enumerate(axes):
    ax.set_xlim(-0.2, 7.5); ax.set_ylim(-0.3, 5.2)
    ax.axis('off'); ax.set_facecolor('white')

    cell = 0.7
    # Draw 5×5 input grid
    img_vals = [[1,0,1,0,1],[0,1,0,1,0],[1,0,1,0,1],[0,1,0,1,0],[1,0,1,0,1]]
    draw_grid(ax, 5, 5, 0, 4.9, cell, vals=img_vals, fontsize=7)

    # Highlight kernel position(s)
    kernel_positions = []
    if ax_idx == 0:   # stride=1, show position 0 and position 1
        kernel_positions = [(0,0,'#e74c3c'), (0,1,'#f39c12')]
        out_size = '3×3 output'
    elif ax_idx == 1: # stride=2, show positions 0 and 2
        kernel_positions = [(0,0,'#e74c3c'), (0,2,'#f39c12')]
        out_size = '2×2 output'
    else:             # padding=1 — show border
        kernel_positions = [(0,0,'#e74c3c')]
        out_size = '5×5 output'
        # Draw padding border
        for r2 in range(-1, 6):
            for c2 in [-1, 5]:
                rect = plt.Rectangle((-0.05 + (c2)*cell - cell*0.5 + cell,
                                      4.9 - r2*cell - cell),
                                     cell, cell,
                                     facecolor='#d4e6f1', edgecolor='#2980b9',
                                     lw=0.8, linestyle='--')
                # simpler: just draw border rects
        # Draw a dashed zero-border around the 5×5 grid
        border = plt.Rectangle((-0.05 - cell + cell, 4.9 - 5*cell - 0.05 + cell),
                                5*cell + 2*0.05, 5*cell + 2*0.05,
                                facecolor='none', edgecolor='#2980b9',
                                lw=1.5, linestyle='--')
        ax.add_patch(border)
        ax.text(5*cell/2, 4.9 - 5*cell - 0.35 + cell, 'zero padding',
                ha='center', fontsize=7, color='#2980b9')

    for (r_k, c_k, col) in kernel_positions:
        for dr in range(3):
            for dc in range(3):
                rect = plt.Rectangle((c_k*cell + dc*cell - 0.01,
                                      4.9 - r_k*cell - dr*cell - cell + 0.01),
                                     cell - 0.02, cell - 0.02,
                                     facecolor=col, edgecolor='white',
                                     lw=1.5, alpha=0.45, zorder=3)
                ax.add_patch(rect)

    # Arrow showing direction of travel
    ax.annotate('', xy=(4.7, 2.55), xytext=(0.0, 2.55),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
    ax.text(2.4, 2.2, 'slides →', ha='center', fontsize=8, color='#555')

    # Output size label
    ax.text(5.5, 3.8, out_size, ha='center', fontsize=9,
            color='#1F3864', fontweight='bold')

    # Legend
    ax.add_patch(plt.Rectangle((5.2, 1.4), 0.4, 0.4, facecolor='#e74c3c', alpha=0.5))
    ax.text(5.7, 1.62, 'position 1', fontsize=7, va='center')
    if len(kernel_positions) > 1:
        ax.add_patch(plt.Rectangle((5.2, 0.8), 0.4, 0.4, facecolor='#f39c12', alpha=0.5))
        ax.text(5.7, 1.02, 'position 2', fontsize=7, va='center')

    ax.set_title(titles[ax_idx], fontsize=9, fontweight='bold',
                 color='#1F3864', pad=6)

fig.suptitle('Figure 5.0a — How stride and padding control the size of the output feature map',
             fontsize=11, fontweight='bold', color='#1F3864', y=1.04)
plt.tight_layout()
plt.show()

print('Key rules:')
print('  Output size = (Input - Kernel + 2×Padding) / Stride  + 1')
print('  Stride=1, padding=1, kernel=3×3  →  (5-3+2)/1+1 = 5  (same size)')
print('  Stride=2, padding=0, kernel=3×3  →  (5-3+0)/2+1 = 2  (halved)')
print()
print('In practice: use padding=1 to keep size constant; use MaxPool2d(2,2) to halve it deliberately.')

## Stride and padding

**Stride** controls how far the kernel jumps between positions. Stride=1 moves one pixel at a time, producing an output nearly the same size as the input. Stride=2 jumps two pixels at a time, which roughly halves the spatial dimensions and reduces computation.

**Padding** adds extra pixels — usually zeros — around the *border of the input image* before the convolution is applied. It does not shrink the kernel, add new kernels, or change what the kernel looks for. The kernel stays exactly the same size; it simply now has more positions to slide over because the input has been extended.

`padding=1` means a one-pixel-wide border of zeros is added all the way around the image:

```
Before padding (5×5):       After padding=1 (7×7):

  A B C D E                 0 0 0 0 0 0 0
  F G H I J                 0 A B C D E 0
  K L M N O       →         0 F G H I J 0
  P Q R S T                 0 K L M N O 0
  U V W X Y                 0 P Q R S T 0
                             0 U V W X Y 0
                             0 0 0 0 0 0 0
```

Why does this matter? Without padding, a 3×3 kernel applied to a 5×5 input produces a 3×3 output — the edges lose one pixel on each side. With `padding=1`, the 3×3 kernel applied to the padded 7×7 input produces a 5×5 output — the same spatial size as the original input. This is essential for building deep networks: without padding, feature maps would shrink after every convolutional layer and quickly become too small.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 5.1 — Convolution intuition
#
# We apply two hand-crafted kernels to a real Fashion-MNIST image:
#   - An edge detection kernel (Sobel-like)
#   - A blur kernel (averaging)
#
# This shows concretely what a convolutional layer 'sees' before any training.
# ─────────────────────────────────────────────────────────────────────────────

import torch.nn.functional as F

# Load one Fashion-MNIST image (no labels needed here)
fmnist_raw = datasets.FashionMNIST('./data', train=True, download=True,
                                    transform=transforms.ToTensor())
img_tensor, label = fmnist_raw[0]   # shape: (1, 28, 28)
class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
                'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Add batch dimension: (1, 1, 28, 28)
img_4d = img_tensor.unsqueeze(0)

# ── Kernel 1: horizontal edge detection ──────────────────────────────────────
# This kernel highlights horizontal transitions (light-to-dark or dark-to-light)
kernel_edge = torch.tensor([[
    [-1., -2., -1.],
    [ 0.,  0.,  0.],
    [ 1.,  2.,  1.]
]]).unsqueeze(0)   # shape: (1, 1, 3, 3) — out_channels, in_channels, H, W

# ── Kernel 2: blur (box filter) ──────────────────────────────────────────────
# Averages each pixel with its neighbours — smooths the image
kernel_blur = torch.ones(1, 1, 3, 3) / 9.0

# ── Kernel 3: vertical edge detection ────────────────────────────────────────
kernel_vert = torch.tensor([[
    [-1., 0., 1.],
    [-2., 0., 2.],
    [-1., 0., 1.]
]]).unsqueeze(0)

# Apply each kernel with padding=1 to keep the same spatial size
with torch.no_grad():
    out_edge = F.conv2d(img_4d, kernel_edge, padding=1).squeeze()
    out_blur = F.conv2d(img_4d, kernel_blur, padding=1).squeeze()
    out_vert = F.conv2d(img_4d, kernel_vert, padding=1).squeeze()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
fig.patch.set_facecolor('white')

axes[0].imshow(img_tensor.squeeze(), cmap='gray')
axes[0].set_title(f'Original\n({class_names[label]})', fontsize=10, fontweight='bold')

axes[1].imshow(out_edge.numpy(), cmap='gray')
axes[1].set_title('Horizontal edge kernel\n(detects top/bottom edges)', fontsize=10)

axes[2].imshow(out_vert.numpy(), cmap='gray')
axes[2].set_title('Vertical edge kernel\n(detects left/right edges)', fontsize=10)

axes[3].imshow(out_blur.numpy(), cmap='gray')
axes[3].set_title('Blur kernel\n(averages neighbours)', fontsize=10)

for ax in axes:
    ax.axis('off')

fig.suptitle('Figure 5.1 — The same image seen through three different kernels\n'
             'In a trained CNN, the network learns these kernels automatically from data',
             fontsize=11, fontweight='bold', color='#1F3864', y=1.08)
plt.tight_layout()
plt.show()

print('Each output is a feature map — a new 28×28 image showing where the kernel activated.')
print('A real CNN layer learns 32 or 64 such kernels simultaneously, each detecting different patterns.')

## Reading Figure 5.1

Each panel shows the same Fashion-MNIST image after being processed by a different 3×3 kernel.

**Horizontal edge kernel:** The output is bright where the image transitions from dark to light (or light to dark) in the vertical direction — the top and bottom edges of the garment are highlighted. Horizontal regions with no transition appear dark.

**Vertical edge kernel:** The same idea but rotated 90°. Left and right edges of the garment light up while horizontal transitions are suppressed.

**Blur kernel:** Every output pixel is the average of its 3×3 neighbourhood. Fine detail is lost and the image appears smoother.

In a real trained CNN, the network discovers kernels like these — and many more complex ones — automatically by minimising the loss function. You do not specify what to look for. The kernels are the weights that are learned through backpropagation, just like the weights in an FFN.

### 📝 Exercise 5.1 — Apply Your Own Kernel

A **sharpening kernel** enhances edges by subtracting a blurred version from the original. The kernel below does this:

```
 0  -1   0
-1   5  -1
 0  -1   0
```

Fill in the kernel values below and observe the output. Then try modifying the centre value and notice what changes.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 5.1 — Apply a sharpening kernel
#
# Replace each None with the correct kernel value (integers), then run the cell.
# ─────────────────────────────────────────────────────────────────────────────

# kernel_sharp = torch.tensor([[
#     [ 0., None,  0.],
#     [None, None, None],
#     [ 0., None,  0.]
# ]]).unsqueeze(0)

# with torch.no_grad():
#     out_sharp = F.conv2d(img_4d, kernel_sharp, padding=1).squeeze()
#
# fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
# fig.patch.set_facecolor('white')
# axes[0].imshow(img_tensor.squeeze(), cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
# axes[1].imshow(out_sharp.numpy(), cmap='gray');   axes[1].set_title('Sharpened'); axes[1].axis('off')
# plt.tight_layout(); plt.show()

# ── Suggested answer — see Answer Key at the end of this chapter ──────────

# Run this cell after attempting the exercise yourself.
# The kernel below is the solution — uncomment to verify your answer.

# kernel_sharp = torch.tensor([[
#     [ 0., -1.,  0.],
#     [-1.,  5., -1.],
#     [ 0., -1.,  0.]
# ]]).unsqueeze(0)
#
# with torch.no_grad():
#     out_sharp = F.conv2d(img_4d, kernel_sharp, padding=1).squeeze()
#
# fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
# fig.patch.set_facecolor('white')
# axes[0].imshow(img_tensor.squeeze(), cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
# axes[1].imshow(out_sharp.numpy(), cmap='gray');   axes[1].set_title('Sharpened'); axes[1].axis('off')
# plt.tight_layout(); plt.show()
# print('The sharpened image has crisper edges. Try changing the centre value from 5 to 9.')

print('Exercise 5.1 — fill in the kernel values above and run.')

---
# 5.2 Pooling and CNN Architecture

## Max Pooling

After a convolutional layer produces feature maps, they are typically large. A 28×28 input with 32 kernels produces 32 feature maps of 28×28 each — 25,088 values. Processing this directly would be expensive, and it is more information than necessary.

**Max pooling** down-samples each feature map by dividing it into non-overlapping regions (typically 2×2) and keeping only the maximum value from each region. This halves the spatial dimensions (28×28 → 14×14), reducing computation and introducing a small amount of translation invariance — the model becomes slightly less sensitive to the exact position of a feature.

Average pooling does the same but takes the mean instead of the maximum. Max pooling is used more often in practice because it tends to preserve the most salient activations.

## The standard CNN building block

A convolutional block follows a consistent pattern:

```
Conv2d  →  ReLU  →  MaxPool2d
```

- **Conv2d** applies the learned kernels to produce feature maps
- **ReLU** introduces non-linearity (same role as in an FFN)
- **MaxPool2d** reduces spatial dimensions and computation

A full CNN stacks several of these blocks, with the number of channels typically growing (e.g. 1→32→64) while spatial dimensions shrink. After the convolutional blocks, the feature maps are flattened into a vector and passed to a fully connected head for the final classification.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 5.0b — Max pooling step by step
# Shows a 4×4 feature map being reduced to 2×2 by MaxPool2d(kernel=2, stride=2)
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
fig.patch.set_facecolor('white')

input_map  = [[3, 1, 2, 8], [5, 6, 9, 4], [7, 2, 1, 3], [4, 8, 6, 5]]
output_map = [[6, 9], [8, 6]]
regions = [
    {'rows': [0,1], 'cols': [0,1], 'color': '#fadbd8', 'max': 6},
    {'rows': [0,1], 'cols': [2,3], 'color': '#d5f5e3', 'max': 9},
    {'rows': [2,3], 'cols': [0,1], 'color': '#d4e6f1', 'max': 8},
    {'rows': [2,3], 'cols': [2,3], 'color': '#fef9e7', 'max': 6},
]
cell = 0.7
for ax in axes:
    ax.set_xlim(-0.3, 5); ax.set_ylim(-0.3, 3.5)
    ax.axis('off'); ax.set_facecolor('white')

# Panel 0 — input with colour-coded 2×2 regions
ax = axes[0]
for reg in regions:
    for r in reg['rows']:
        for c in reg['cols']:
            bold = (input_map[r][c] == reg['max'])
            ax.add_patch(plt.Rectangle((c*cell, 3.2 - r*cell - cell), cell, cell,
                         facecolor=reg['color'], edgecolor='#888', lw=1.0))
            ax.text(c*cell + cell/2, 3.2 - r*cell - cell/2,
                    str(input_map[r][c]), ha='center', va='center', fontsize=11,
                    fontweight='bold' if bold else 'normal',
                    color='#c0392b' if bold else '#1a1a1a')
ax.set_title('Input feature map (4×4)\nColour = 2×2 region  |  Red bold = max', fontsize=9, fontweight='bold', color='#1F3864')

# Panel 1 — operation label
ax = axes[1]
ax.text(1.5, 1.8, 'MaxPool2d\n(kernel=2, stride=2)\n\nKeep the maximum\nfrom each 2×2 region',
        ha='center', va='center', fontsize=10, color='#1F3864',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#eef2f9', edgecolor='#2980b9'))

# Panel 2 — output 2×2
ax = axes[2]
region_colors = ['#fadbd8', '#d5f5e3', '#d4e6f1', '#fef9e7']
for i, (r, c) in enumerate([(0,0),(0,1),(1,0),(1,1)]):
    ax.add_patch(plt.Rectangle((c*cell + 0.5, 2.0 - r*cell), cell, cell,
                 facecolor=region_colors[i], edgecolor='#888', lw=1.0))
    ax.text(c*cell + 0.5 + cell/2, 2.0 - r*cell + cell/2,
            str(output_map[r][c]), ha='center', va='center', fontsize=13,
            fontweight='bold', color='#c0392b')
ax.set_title('Output feature map (2×2)\nSpatial size halved', fontsize=9, fontweight='bold', color='#1F3864')

fig.suptitle('Figure 5.0b — Max Pooling: 4×4 → 2×2  (one 2×2 region collapses to its maximum)',
             fontsize=11, fontweight='bold', color='#1F3864', y=1.06)
plt.tight_layout()
plt.show()
print('The maximum in each region indicates whether a pattern was present.')
print('Exactly where it was within the region matters less — this is translation invariance.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 5.2 — Anatomy of a 2-block CNN
# Shows how spatial dimensions and channel counts change through the network.
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14); ax.set_ylim(0, 5)
ax.axis('off'); fig.patch.set_facecolor('white'); ax.set_facecolor('white')

stages = [
    {'x': 1.0, 'label': 'Input',         'sub': '1 × 28 × 28',   'detail': '(C × H × W)',    'color': '#d4e6f1', 'edge': '#2980b9'},
    {'x': 3.5, 'label': 'Conv2d\n+ ReLU','sub': '32 × 28 × 28',  'detail': '32 kernels 3×3\npadding=1', 'color': '#d5f5e3', 'edge': '#27ae60'},
    {'x': 5.5, 'label': 'MaxPool2d',     'sub': '32 × 14 × 14',  'detail': '2×2 pool\nstride=2',       'color': '#fef9e7', 'edge': '#f39c12'},
    {'x': 7.5, 'label': 'Conv2d\n+ ReLU','sub': '64 × 14 × 14',  'detail': '64 kernels 3×3\npadding=1', 'color': '#d5f5e3', 'edge': '#27ae60'},
    {'x': 9.5, 'label': 'MaxPool2d',     'sub': '64 × 7 × 7',    'detail': '2×2 pool\nstride=2',       'color': '#fef9e7', 'edge': '#f39c12'},
    {'x': 11.2,'label': 'Flatten',       'sub': '3136',           'detail': '64 × 7 × 7 = 3136',        'color': '#fdebd0', 'edge': '#e67e22'},
    {'x': 13.0,'label': 'Linear\n+ class','sub': '10 classes',   'detail': 'FC head',                   'color': '#f9e79f', 'edge': '#d4ac0d'},
]

for s in stages:
    rect = mpatches.FancyBboxPatch((s['x']-0.85, 1.3), 1.7, 2.4,
                                    boxstyle='round,pad=0.1',
                                    facecolor=s['color'], edgecolor=s['edge'], lw=2)
    ax.add_patch(rect)
    ax.text(s['x'], 3.3, s['label'],  ha='center', va='center', fontsize=9,  fontweight='bold', color='#1a1a1a')
    ax.text(s['x'], 2.5, s['sub'],    ha='center', va='center', fontsize=9,  color='#1a5276', fontweight='bold')
    ax.text(s['x'], 1.8, s['detail'], ha='center', va='center', fontsize=8,  color='#444', linespacing=1.4)

# Arrows between stages
for i in range(len(stages)-1):
    x1 = stages[i]['x']   + 0.85
    x2 = stages[i+1]['x'] - 0.85
    ax.annotate('', xy=(x2, 2.5), xytext=(x1, 2.5),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.8))

# Brace labels for blocks
ax.text(4.5, 0.7, '── Block 1 ──', ha='center', fontsize=9, color='#27ae60', fontweight='bold')
ax.text(8.5, 0.7, '── Block 2 ──', ha='center', fontsize=9, color='#27ae60', fontweight='bold')

ax.set_title('Figure 5.2 — 2-Block CNN Architecture for Fashion-MNIST\n'
             'Channels grow (1→32→64) while spatial dimensions shrink (28→14→7)',
             fontsize=11, fontweight='bold', color='#1F3864', pad=12)
plt.tight_layout()
plt.show()

## From feature maps to a classification decision

After the convolutional blocks have processed the image, the network holds a set of feature maps — in this case, 64 maps each of size 7×7. These are still spatially arranged grids, not yet a prediction. Two more steps turn them into class probabilities:

**Flatten.** The 64 × 7 × 7 feature maps are reshaped into a single 1-D vector of 3,136 numbers (64 × 7 × 7 = 3,136). This is purely a reshape operation — no weights, no computation, no information loss. It simply unrolls all the feature values into a format that a fully connected layer can accept.

**Linear (fully connected) layer.** A standard `nn.Linear` layer — identical in structure to the layers used in Chapter 4's FFN — takes the 3,136-element vector and maps it to a smaller representation (256 neurons here). This layer learns which combinations of spatial features are important for the final decision. A ReLU activation and Dropout follow for non-linearity and regularisation.

**Classification layer.** A final `nn.Linear` maps the 256 features to 10 output values — one per class. These raw values (called *logits*) are passed to `CrossEntropyLoss`, which applies Softmax internally to convert them into probabilities. The predicted class is the one with the highest probability.

This combination — convolutional blocks for spatial feature extraction, then a small FFN head for classification — is the standard pattern for image classifiers throughout this chapter and in most production vision systems.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Define the CNN for Fashion-MNIST
#
# Constructor kwargs match model_config keys exactly —
# same contract as Chapter 3 (CaliforniaNet) and Chapter 4 (FFN).
# ─────────────────────────────────────────────────────────────────────────────

class FashionCNN(nn.Module):
    """2-block CNN for Fashion-MNIST (grayscale 28×28, 10 classes).
    Constructor kwargs match model_config for ModelPipeline compatibility.
    """
    def __init__(self, num_classes=10, dropout_p=0.5):
        super().__init__()

        # ── Convolutional blocks ───────────────────────────────────────────────
        self.features = nn.Sequential(
            # Block 1: 1 × 28 × 28  →  32 × 14 × 14
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # same-size output
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),        # halves H and W

            # Block 2: 32 × 14 × 14  →  64 × 7 × 7
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Fully connected classifier head ───────────────────────────────────
        # After 2 pooling layers: 64 channels × 7 × 7 = 3136
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


sample = FashionCNN()
print(sample)
print(f'\nTotal trainable parameters: {count_parameters(sample):,}')
print()

# Trace the shape through the network
dummy = torch.zeros(1, 1, 28, 28)
with torch.no_grad():
    after_block1 = sample.features[:3](dummy)
    after_block2 = sample.features(dummy)
    output       = sample(dummy)

print('Shape trace through the network:')
print(f'  Input          : {tuple(dummy.shape)}')
print(f'  After block 1  : {tuple(after_block1.shape)}')
print(f'  After block 2  : {tuple(after_block2.shape)}')
print(f'  After flatten  : {after_block2.view(1, -1).shape}')
print(f'  Output (logits): {tuple(output.shape)}')

### 📝 Exercise 5.2 — Build a Deeper CNN

Extend `FashionCNN` to a 3-block version by adding a third convolutional block. After three max-pool layers on a 28×28 input, the spatial dimensions will be 3×3.

Fill in the blank values and run the cell to verify the shape trace.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 5.2 — 3-block CNN
#
# Replace each None with the correct value, then run the shape trace.
# Hint: after 3 max-pool(2,2) layers, a 28×28 image becomes 3×3.
# ─────────────────────────────────────────────────────────────────────────────

# class DeepFashionCNN(nn.Module):
#     def __init__(self, num_classes=10, dropout_p=0.5):
#         super().__init__()
#         self.features = nn.Sequential(
#             nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
#             nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
#             nn.Conv2d(64, None, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
#         )
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(None * 3 * 3, 256),  # what is None?
#             nn.ReLU(), nn.Dropout(dropout_p),
#             nn.Linear(256, num_classes)
#         )
#     def forward(self, x): return self.classifier(self.features(x))
#
# m = DeepFashionCNN()
# dummy = torch.zeros(1, 1, 28, 28)
# with torch.no_grad(): print(f'Output shape: {m(dummy).shape}')
# print(f'Parameters: {count_parameters(m):,}')

# ── Student work area ─────────────────────────────────────────────────────
# Uncomment and complete the class definition below, then run the shape trace.

# class DeepFashionCNN(nn.Module):
#     def __init__(self, num_classes=10, dropout_p=0.5):
#         super().__init__()
#         self.features = nn.Sequential(
#             nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
#             nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
#             nn.Conv2d(64, None, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
#         )
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(None * 3 * 3, 256),  # what should None be?
#             nn.ReLU(), nn.Dropout(dropout_p),
#             nn.Linear(256, num_classes)
#         )
#     def forward(self, x): return self.classifier(self.features(x))
#
# m = DeepFashionCNN()
# dummy = torch.zeros(1, 1, 28, 28)
# with torch.no_grad(): print(f'Output shape: {m(dummy).shape}')
# print(f'Parameters: {count_parameters(m):,}')

print('Exercise 5.2 — complete the class above, then uncomment and run.')
print('See the Answer Key at the end of this chapter.')

---
# 5.3 Training and Evaluation

## Loading Fashion-MNIST

**Fashion-MNIST** is a drop-in replacement for MNIST that uses grayscale images of clothing items (T-shirts, trousers, dresses, etc.) instead of handwritten digits. It has the same 60,000 training / 10,000 test structure and 10 classes, but is significantly harder — a well-tuned FFN achieves around 88% accuracy, while a CNN typically reaches 91–93%.

## Data augmentation

Training on the same images repeatedly causes overfitting. **Data augmentation** artificially expands the training set by applying random transformations — flips, rotations, crops — to each image every time it is loaded. The model sees a slightly different version of each image in each epoch, which acts as a regulariser.

Augmentation is applied **only to the training set**. The validation and test sets use clean, unmodified images so evaluation remains consistent across runs.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load Fashion-MNIST with and without augmentation
# ─────────────────────────────────────────────────────────────────────────────

# Validation/test transform — clean images only
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

# Training transform — augmentation applied randomly at load time
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),       # 50% chance of flipping
    transforms.RandomRotation(degrees=10),         # rotate up to ±10°
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # small shift
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

fmnist_train = datasets.FashionMNIST('./data', train=True,  download=True, transform=train_transform)
fmnist_test  = datasets.FashionMNIST('./data', train=False, download=True, transform=val_transform)

fmnist_train_loader = DataLoader(fmnist_train, batch_size=128, shuffle=True,  num_workers=0)
fmnist_val_loader   = DataLoader(fmnist_test,  batch_size=128, shuffle=False, num_workers=0)

CLASS_NAMES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal',  'Shirt',   'Sneaker',  'Bag',   'Ankle boot']

print(f'Training samples : {len(fmnist_train):,}')
print(f'Test samples     : {len(fmnist_test):,}')
print(f'Classes          : {CLASS_NAMES}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 5.3 — Augmentation examples
#
# We extract one PIL image from the dataset, then apply the training transform
# 8 separate times. Because each transform includes random operations, each
# application produces a different result — showing the variety the model sees.
#
# The bottom row shows 8 different clean val images for comparison.
# ─────────────────────────────────────────────────────────────────────────────

# Get the raw (un-transformed) dataset to extract a single PIL image
fmnist_pil_ds = datasets.FashionMNIST('./data', train=True, download=False, transform=None)
pil_img, img_label = fmnist_pil_ds[0]   # PIL Image, no transforms applied yet

fig, axes = plt.subplots(2, 8, figsize=(15, 4))
fig.patch.set_facecolor('white')

for i in range(8):
    # Top row: apply train_transform fresh each time — random ops give different result
    torch.manual_seed(i * 17)  # different seed per column for visible variety
    np.random.seed(i * 17)
    augmented = train_transform(pil_img)  # returns tensor (1, 28, 28)
    axes[0, i].imshow(augmented.squeeze() * 0.5 + 0.5, cmap='gray')  # un-normalise
    axes[0, i].axis('off')
    axes[0, i].set_title(f'Aug {i+1}', fontsize=8, color='#2980b9')

    # Bottom row: 8 different clean images from the validation set
    img_clean, lbl_c = fmnist_test[i]
    axes[1, i].imshow(img_clean.squeeze() * 0.5 + 0.5, cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title(CLASS_NAMES[lbl_c], fontsize=8, color='#555')

# Left-side row labels
axes[0, 0].set_ylabel(f'TOP ROW\nSame {CLASS_NAMES[img_label]}\n8× augmented',
                      fontsize=8, rotation=90, labelpad=4, color='#2980b9')
axes[1, 0].set_ylabel('BOTTOM ROW\n8 different\nval images',
                      fontsize=8, rotation=90, labelpad=4, color='#555')

fig.suptitle('Figure 5.3 — Data augmentation: the same image, 8 different random transforms\n'
             'Each epoch the model sees a slightly different version — this prevents memorisation',
             fontsize=11, fontweight='bold', color='#1F3864', y=1.04)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Train Fashion-MNIST CNN
#
# Optimiser: SGD + Momentum — recommended for CNNs in Chapter 3 (Table 3.2).
# Momentum accumulates velocity in consistent gradient directions, which helps
# CNNs navigate the ravine-shaped loss surfaces common in image tasks.
# ─────────────────────────────────────────────────────────────────────────────

fmnist_config = {
    'num_classes': 10,
    'dropout_p'  : 0.5,
}

torch.manual_seed(42)
fmnist_model = FashionCNN(**fmnist_config).to(device)
fmnist_crit  = nn.CrossEntropyLoss()

# SGD + Momentum as recommended in Chapter 3 for CNNs
fmnist_opt   = optim.SGD(fmnist_model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

# CosineAnnealingLR: smoothly decays LR over all epochs — well-suited to SGD+Momentum
fmnist_sched = optim.lr_scheduler.CosineAnnealingLR(fmnist_opt, T_max=20)
fmnist_es    = EarlyStopping(patience=8)

print('Training FashionCNN (up to 20 epochs, SGD + Momentum):')
fmnist_train_hist, fmnist_val_hist = run_training(
    model         = fmnist_model,
    train_loader  = fmnist_train_loader,
    val_loader    = fmnist_val_loader,
    criterion     = fmnist_crit,
    optimiser     = fmnist_opt,
    num_epochs    = 20,
    device        = device,
    scheduler     = fmnist_sched,   # CosineAnnealingLR — epoch-based
    early_stopping= fmnist_es,
    verbose_every = 1,
)
print('\nTraining complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 5.4 — Evaluation: loss curves, confusion matrix, learned filters
# ─────────────────────────────────────────────────────────────────────────────

# Collect predictions
fmnist_model.eval()
all_preds, all_labels = [], []
correct = total = 0
with torch.no_grad():
    for imgs, labels in fmnist_val_loader:
        logits = fmnist_model(imgs.to(device))
        preds  = logits.argmax(dim=1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

test_acc = 100.0 * correct / total

fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('white')

# (a) Loss curves
ax1 = fig.add_subplot(2, 3, 1)
ep = range(1, len(fmnist_train_hist)+1)
ax1.plot(ep, fmnist_train_hist, color='#2980b9', lw=2.5, label='Train')
ax1.plot(ep, fmnist_val_hist,   color='#e74c3c', lw=2.5, linestyle='--', label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('(a) Loss curves', fontweight='bold', color='#1F3864')
ax1.legend()
ax1.text(0.98, 0.95, f'Test acc: {test_acc:.1f}%', transform=ax1.transAxes,
         ha='right', va='top', fontsize=10, color='#27ae60', fontweight='bold')

# (b) Confusion matrix
ax2 = fig.add_subplot(2, 3, (2, 3))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.3, ax=ax2, cbar=False, annot_kws={'size': 7})
ax2.set_xlabel('Predicted', fontsize=10); ax2.set_ylabel('Actual', fontsize=10)
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0, fontsize=8)
ax2.set_title('(b) Confusion matrix — test set', fontweight='bold', color='#1F3864')

# (c) Learned filters from the first convolutional layer
filters = fmnist_model.features[0].weight.data.cpu()  # shape: (32, 1, 3, 3)
for i in range(16):
    ax = fig.add_subplot(4, 8, 17 + i)
    f = filters[i, 0].numpy()
    # Normalise each filter for display
    f = (f - f.min()) / (f.max() - f.min() + 1e-8)
    ax.imshow(f, cmap='RdBu_r', vmin=0, vmax=1)
    ax.axis('off')

fig.text(0.5, 0.27, '(c) First 16 learned 3×3 filters from Conv layer 1 — each detects a different local pattern',
         ha='center', fontsize=9, color='#1F3864', fontweight='bold')

fig.suptitle('Figure 5.4 — Fashion-MNIST CNN: Full Evaluation',
             fontsize=13, fontweight='bold', color='#1F3864', y=1.01)
plt.tight_layout()
plt.show()

print(f'Test accuracy: {test_acc:.2f}%')
print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

## Reading Figure 5.4

### Panel (a) — Loss curves
Both training and validation loss fall together, which indicates that the model is generalising rather than memorising the training set. The gap between train and val loss is small, suggesting augmentation is working as a regulariser. The test accuracy is printed in the top-right corner.

### Panel (b) — Confusion matrix
The diagonal holds correct predictions; off-diagonal cells are mistakes. The most common errors are typically between visually similar classes: **Shirt vs T-shirt** (both are upper-body garments with similar shapes), **Coat vs Pullover**, and **Sneaker vs Ankle boot**. These are the same confusions a human might make with a blurry 28×28 image.

### Panel (c) — Learned filters
These are the actual 3×3 kernel weights learned by the first convolutional layer. Unlike the hand-crafted kernels in Figure 5.1, these were discovered entirely by gradient descent. You can see that many filters resemble edge detectors — some are diagonal, some horizontal, some vertical. Others detect combinations of light and dark that do not have an obvious name. This is what "automatic feature learning" looks like in practice.

## Reading the Classification Report

The `classification_report` printed after Figure 5.4 has one row per class and four columns. Here is how to read each number:

| Column | What it measures | How to read it |
|--------|-----------------|----------------|
| **precision** | Of all images the model predicted as this class, what fraction actually belong to it? | High precision = few false alarms for this class |
| **recall** | Of all images that actually belong to this class, what fraction did the model correctly find? | High recall = the model rarely misses this class |
| **f1-score** | The harmonic mean of precision and recall — a single number balancing both | Useful when precision and recall are both important |
| **support** | How many test images belong to this class | Larger support = more reliable estimate |

The bottom three rows summarise across all classes:
- **accuracy** — overall percentage of correct predictions
- **macro avg** — plain average of each metric across all 10 classes (treats every class equally)
- **weighted avg** — average weighted by class size (gives more weight to frequent classes)

**What to look for:** classes with low recall (the model misses them often) or low precision (it over-predicts them) are worth investigating. In Fashion-MNIST, `Shirt` and `T-shirt/Top` typically have the lowest scores because they look similar — the model struggles to distinguish them.

### 📝 Exercise 5.3 — Measure the Effect of Augmentation

Train the same CNN **without** data augmentation and compare the validation accuracy and the train/val loss gap. A larger gap between training loss and validation loss is a sign of overfitting.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 5.3 — Train without augmentation and compare
#
# The cell is ready to run — no values to fill in.
# Compare final val accuracy and the train/val gap to the augmented model above.
# ─────────────────────────────────────────────────────────────────────────────

fmnist_no_aug = datasets.FashionMNIST('./data', train=True,  download=False, transform=val_transform)
loader_no_aug = DataLoader(fmnist_no_aug, batch_size=128, shuffle=True, num_workers=0)

torch.manual_seed(42)
model_no_aug  = FashionCNN(**fmnist_config).to(device)
opt_no_aug    = optim.SGD(model_no_aug.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
sched_no_aug  = optim.lr_scheduler.CosineAnnealingLR(opt_no_aug, T_max=20)
es_no_aug     = EarlyStopping(patience=8)

print('Training without augmentation:')
train_hist_na, val_hist_na = run_training(
    model         = model_no_aug,
    train_loader  = loader_no_aug,
    val_loader    = fmnist_val_loader,
    criterion     = fmnist_crit,
    optimiser     = opt_no_aug,
    num_epochs    = 20,
    device        = device,
    scheduler     = sched_no_aug,
    early_stopping= es_no_aug,
    verbose_every = 5,
)

# Compare final gap
aug_gap    = fmnist_val_hist[-1]  - fmnist_train_hist[-1]
no_aug_gap = val_hist_na[-1]      - train_hist_na[-1]

print(f'\nWith augmentation    — final train loss: {fmnist_train_hist[-1]:.4f}  '
      f'val loss: {fmnist_val_hist[-1]:.4f}  gap: {aug_gap:.4f}')
print(f'Without augmentation — final train loss: {train_hist_na[-1]:.4f}  '
      f'val loss: {val_hist_na[-1]:.4f}  gap: {no_aug_gap:.4f}')
print()
print('A larger train/val gap without augmentation indicates overfitting.')
print('Augmentation acts as a regulariser by preventing the model from memorising exact images.')

---
# 5.4 Business Application: Manufacturing Defect Detection

## The business problem

Manufacturing quality control is one of the clearest industrial applications of computer vision. Traditionally, inspectors examine products by eye on a production line — a process that is slow, expensive, and prone to fatigue-related errors. A CNN trained on labelled images of conforming and defective products can automate this inspection, operating at line speed with consistent accuracy.

The **Casting Product Defect Images** dataset from Kaggle contains photographs of pump impeller castings used in industrial machinery. Each image is labelled as either **OK** (no defect) or **Defective**. The task is binary image classification.

## Why the cost of errors matters here

In a manufacturing context, the two error types have very different consequences:

- **False negative** (missing a defect): a defective part ships to a customer. Consequences can include equipment failure, safety incidents, and costly recalls.
- **False positive** (flagging a good part): a conforming part is scrapped or sent for manual re-inspection. This wastes material and slows production.

In most manufacturing contexts, false negatives are far more expensive than false positives. The model should therefore err on the side of caution — we will tune the decision threshold accordingly, just as we did for churn in Chapter 4.

## Downloading the dataset

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Kaggle authentication — same pattern as Chapter 4
# ─────────────────────────────────────────────────────────────────────────────

kaggle_token    = getpass.getpass('Paste your Kaggle API token (KGAT_...): ')
kaggle_username = input('Enter your Kaggle username: ')

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump({'username': kaggle_username.strip(), 'key': kaggle_token.strip()}, f)
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

!kaggle datasets download -d ravirajsinh45/real-life-industrial-dataset-of-casting-product --unzip -p ./data/casting
print('Download complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Explore the dataset structure
#
# The Kaggle dataset may unzip into slightly different folder structures
# depending on the version. This cell detects the correct path automatically.
# ─────────────────────────────────────────────────────────────────────────────

import pathlib

def find_casting_root(base='./data/casting'):
    """Search for the casting_data directory under any depth below base."""
    base_path = pathlib.Path(base)
    # Look for a directory that contains train/ and test/ sub-folders
    for candidate in sorted(base_path.rglob('*')):
        if candidate.is_dir():
            has_train = (candidate / 'train').exists()
            has_test  = (candidate / 'test').exists()
            if has_train and has_test:
                return candidate
    return None

data_root = find_casting_root('./data/casting')

if data_root is None:
    # Try a flat layout — some versions unzip directly into train/ test/
    data_root = pathlib.Path('./data/casting')
    if not (data_root / 'train').exists():
        print('WARNING: Could not find casting dataset. Expected structure:')
        print('  ./data/casting/  ->  train/  test/')
        print('  with sub-folders ok_front/ and def_front/ inside each.')
        print('Re-run the download cell above and try again.')
    else:
        print(f'Found casting data at: {data_root}')
else:
    print(f'Found casting data at: {data_root}')

# Print image counts per split/class
IMG_EXTENSIONS = {'.jpeg', '.jpg', '.png', '.bmp'}

def count_images(folder):
    return sum(1 for f in folder.rglob('*') if f.suffix.lower() in IMG_EXTENSIONS)

for split in ['train', 'test']:
    for cls in ['ok_front', 'def_front']:
        path = data_root / split / cls
        if path.exists():
            label = 'OK' if 'ok' in cls else 'Defective'
            print(f'  {split:5s} / {label:<10} : {count_images(path):,} images')

# Figure 5.5 — sample images
def get_images(folder, n=4):
    imgs = [f for f in folder.rglob('*') if f.suffix.lower() in IMG_EXTENSIONS]
    return sorted(imgs)[:n]

ok_paths  = get_images(data_root / 'train' / 'ok_front',  n=4)
def_paths = get_images(data_root / 'train' / 'def_front', n=4)

if not ok_paths or not def_paths:
    print('No images found — check that the dataset was downloaded and unzipped correctly.')
else:
    fig, axes = plt.subplots(2, 4, figsize=(13, 6))
    fig.patch.set_facecolor('white')
    for i, p in enumerate(ok_paths):
        img = Image.open(p).convert('RGB')   # convert ensures displayable image
        axes[0, i].imshow(img)
        axes[0, i].set_title('OK', fontsize=9, color='#27ae60', fontweight='bold')
        axes[0, i].axis('off')
    for i, p in enumerate(def_paths):
        img = Image.open(p).convert('RGB')
        axes[1, i].imshow(img)
        axes[1, i].set_title('Defective', fontsize=9, color='#e74c3c', fontweight='bold')
        axes[1, i].axis('off')
    fig.suptitle('Figure 5.5 — Sample casting images: OK (top) vs Defective (bottom)\n'
                 'Defects appear as voids, cracks, or surface irregularities',
                 fontsize=11, fontweight='bold', color='#1F3864', y=1.04)
    plt.tight_layout()
    plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Transforms and DataLoaders for the defect dataset
#
# Images are grayscale and resized to 64×64 for efficiency on CPU.
# The transforms are saved as part of the ModelPipeline in Section 5.5.
# ─────────────────────────────────────────────────────────────────────────────

DEFECT_SIZE   = 64    # resize all images to 64×64
DEFECT_MEAN   = (0.5,)
DEFECT_STD    = (0.5,)

defect_train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((DEFECT_SIZE, DEFECT_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(DEFECT_MEAN, DEFECT_STD),
])

defect_val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((DEFECT_SIZE, DEFECT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(DEFECT_MEAN, DEFECT_STD),
])

train_dir = str(data_root / 'train')
test_dir  = str(data_root / 'test')

defect_train_ds = datasets.ImageFolder(train_dir, transform=defect_train_transform)
defect_test_ds  = datasets.ImageFolder(test_dir,  transform=defect_val_transform)

defect_train_loader = DataLoader(defect_train_ds, batch_size=64, shuffle=True,  num_workers=0)
defect_val_loader   = DataLoader(defect_test_ds,  batch_size=64, shuffle=False, num_workers=0)

# ImageFolder assigns labels alphabetically: def_front=0, ok_front=1
DEFECT_CLASS_MAP = defect_train_ds.class_to_idx   # {'def_front': 0, 'ok_front': 1}
DEFECT_LABEL_MAP = {v: k for k, v in DEFECT_CLASS_MAP.items()}
# Rename for readability
DEFECT_LABEL_MAP = {0: 'Defective', 1: 'OK'}

print(f'Training images : {len(defect_train_ds):,}')
print(f'Test images     : {len(defect_test_ds):,}')
print(f'Class mapping   : {DEFECT_LABEL_MAP}')

# Check class balance
train_labels = [defect_train_ds[i][1] for i in range(len(defect_train_ds))]
from collections import Counter
counts = Counter(train_labels)
for cls_idx, cnt in counts.items():
    print(f'  {DEFECT_LABEL_MAP[cls_idx]:<12}: {cnt:,} ({100*cnt/len(defect_train_ds):.1f}%)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Define the defect detection CNN
#
# Input: 1 × 64 × 64 (grayscale, resized)
# Three convolutional blocks — larger input than Fashion-MNIST warrants
# an extra block to capture higher-level defect patterns.
# Output: 2 logits (Defective / OK)
# ─────────────────────────────────────────────────────────────────────────────

class DefectCNN(nn.Module):
    """3-block CNN for binary casting defect detection.
    Input: 1 × 64 × 64. Output: 2 logits (Defective=0, OK=1).
    Constructor kwargs match model_config for ModelPipeline compatibility.
    """
    def __init__(self, num_classes=2, dropout_p=0.5):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1: 1 × 64 × 64  →  32 × 32 × 32
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 2: 32 × 32 × 32  →  64 × 16 × 16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 3: 64 × 16 × 16  →  128 × 8 × 8
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        # After 3 max-pool layers: 128 × 8 × 8 = 8192
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256), nn.ReLU(), nn.Dropout(p=dropout_p),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


defect_config = {
    'num_classes': 2,
    'dropout_p'  : 0.5,
}

torch.manual_seed(42)
defect_model = DefectCNN(**defect_config).to(device)
defect_crit  = nn.CrossEntropyLoss()
defect_opt   = optim.SGD(defect_model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
defect_sched = optim.lr_scheduler.CosineAnnealingLR(defect_opt, T_max=20)
defect_es    = EarlyStopping(patience=8)

print(defect_model)
print(f'\nTrainable parameters: {count_parameters(defect_model):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Train the defect detector
# ─────────────────────────────────────────────────────────────────────────────

print('Training DefectCNN (up to 20 epochs):')
defect_train_hist, defect_val_hist = run_training(
    model         = defect_model,
    train_loader  = defect_train_loader,
    val_loader    = defect_val_loader,
    criterion     = defect_crit,
    optimiser     = defect_opt,
    num_epochs    = 20,
    device        = device,
    scheduler     = defect_sched,
    early_stopping= defect_es,
    verbose_every = 1,
)
print('\nTraining complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Collect test set predictions
# ─────────────────────────────────────────────────────────────────────────────

defect_model.eval()
d_preds, d_probs, d_labels = [], [], []

with torch.no_grad():
    for imgs, labels in defect_val_loader:
        logits = defect_model(imgs.to(device))
        # P(Defective) = P(class=0) — class 0 is the positive/alarm class
        probs  = torch.softmax(logits, dim=1)[:, 0].cpu()
        preds  = logits.argmax(dim=1).cpu()
        d_preds.extend(preds.numpy())
        d_probs.extend(probs.numpy())
        d_labels.extend(labels.numpy())

d_preds  = np.array(d_preds)
d_probs  = np.array(d_probs)
d_labels = np.array(d_labels)

# Invert labels for ROC: 1 = Defective (positive class), 0 = OK
d_labels_roc = 1 - d_labels   # ImageFolder: 0=Defective, 1=OK → flip for convention

print('Classification Report:')
print(classification_report(d_labels, d_preds,
                             target_names=[DEFECT_LABEL_MAP[0], DEFECT_LABEL_MAP[1]]))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 5.6 — Full evaluation dashboard
# ─────────────────────────────────────────────────────────────────────────────

FN_COST = 500   # defective part reaches customer — much more costly
FP_COST = 20    # conforming part scrapped or re-inspected

sweep_thresholds = np.linspace(0.05, 0.95, 90)
total_costs = []
for t in sweep_thresholds:
    preds_t = (d_probs >= t).astype(int)   # 1 = flag as Defective
    # d_labels: 0=Defective(pos), 1=OK(neg)  →  convert to standard: 1=pos
    true_t  = 1 - d_labels
    cm_t    = confusion_matrix(true_t, preds_t, labels=[0, 1])
    tn, fp, fn, tp = cm_t.ravel()
    total_costs.append(fn * FN_COST + fp * FP_COST)

best_t_idx = np.argmin(total_costs)
best_t     = sweep_thresholds[best_t_idx]
best_cost  = total_costs[best_t_idx]

fpr, tpr, _ = roc_curve(d_labels_roc, d_probs)
roc_auc     = auc(fpr, tpr)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.patch.set_facecolor('white')

# (a) Loss curves
ax = axes[0, 0]
ep = range(1, len(defect_train_hist)+1)
ax.plot(ep, defect_train_hist, color='#2980b9', lw=2.5, label='Train')
ax.plot(ep, defect_val_hist,   color='#e74c3c', lw=2.5, linestyle='--', label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('(a) Loss curves', fontweight='bold', color='#1F3864')
ax.legend()

# (b) Confusion matrix
ax = axes[0, 1]
sns.heatmap(confusion_matrix(d_labels, d_preds), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Defective', 'OK'],
            yticklabels=['Defective', 'OK'], linewidths=1, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('(b) Confusion matrix — test set', fontweight='bold', color='#1F3864')

# (c) ROC curve
ax = axes[1, 0]
ax.plot(fpr, tpr, color='#8e44ad', lw=2.5, label=f'AUC = {roc_auc:.3f}')
ax.plot([0,1],[0,1], color='#aaa', lw=1.5, linestyle='--', label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('(c) ROC curve', fontweight='bold', color='#1F3864')
ax.legend(loc='lower right')

# (d) Cost analysis
ax = axes[1, 1]
ax.plot(sweep_thresholds, total_costs, color='#e67e22', lw=2.5)
ax.axvline(best_t, color='#c0392b', lw=2, linestyle='--')
ax.scatter([best_t], [best_cost], color='#c0392b', s=120, zorder=5)
ax.annotate(
    f'Optimal threshold: {best_t:.2f}\nTotal cost: ${best_cost:,}',
    xy=(best_t, best_cost), xytext=(best_t+0.08, best_cost+200),
    fontsize=9, color='#c0392b',
    arrowprops=dict(arrowstyle='->', color='#c0392b'))
ax.set_xlabel('Decision threshold (P(Defective))')
ax.set_ylabel('Total business cost ($)')
ax.set_title(f'(d) Cost analysis  (FN=${FN_COST}, FP=${FP_COST})',
             fontweight='bold', color='#1F3864')

fig.suptitle('Figure 5.6 — Defect Detector: Full Evaluation Dashboard',
             fontsize=14, fontweight='bold', color='#1F3864', y=1.01)
plt.tight_layout()
plt.show()
print(f'ROC AUC: {roc_auc:.4f}')
print(f'Business-optimal threshold: {best_t:.2f}  |  Estimated total cost: ${best_cost:,}')

## Reading Figure 5.6

### Panel (a) — Loss curves
Both curves should fall and converge. A model that learns quickly and shows small train/val gap is benefiting from augmentation and dropout regularisation.

### Panel (b) — Confusion matrix
The most critical cell is the **bottom-left** — actual Defective parts predicted as OK (False Negatives). These are the parts that pass inspection and reach customers. The threshold will be tuned in panel (d) to minimise these.

### Panel (c) — ROC curve
For safety-critical inspection tasks, an AUC above 0.95 is typically expected before deployment. The curve shows whether the model can cleanly separate defective from conforming images at any threshold.

### Panel (d) — Cost analysis
The cost curve reflects the manufacturing reality: a missed defect ($500) is far more expensive than a false alarm ($20). The optimal threshold is therefore shifted **lower** than 0.5 — the model should flag a part as defective at a lower confidence level, accepting more false alarms to ensure fewer defective parts escape. This is the key operational difference between a manufacturing model and a churn model.

### 📝 Exercise 5.4 — Tune the Threshold

The cost analysis found the business-optimal threshold. Apply it and compare the confusion matrix at the default threshold (0.5) versus the optimal threshold.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 5.4 — Compare confusion matrices at two thresholds
#
# Fill in the threshold values then run. Which threshold catches more defects?
# ─────────────────────────────────────────────────────────────────────────────

# threshold_default = None   # replace: what is the standard default threshold?
# threshold_optimal = None   # replace: use best_t computed in Figure 5.6

# Once you have set both values, uncomment and run the plotting code below:

# fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
# fig.patch.set_facecolor('white')
# true_binary = 1 - d_labels  # 1=Defective, 0=OK
# for ax, t, title_suffix in zip(
#     axes,
#     [threshold_default, threshold_optimal],
#     [f'Default threshold = {threshold_default:.2f}',
#      f'Optimal threshold = {threshold_optimal:.2f}']
# ):
#     preds_t = (d_probs >= t).astype(int)
#     cm_t    = confusion_matrix(true_binary, preds_t, labels=[0, 1])
#     sns.heatmap(cm_t, annot=True, fmt='d', cmap='Blues',
#                 xticklabels=['OK', 'Defective'],
#                 yticklabels=['OK', 'Defective'], linewidths=1, ax=ax)
#     ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
#     tn, fp, fn, tp = cm_t.ravel()
#     cost = fn * FN_COST + fp * FP_COST
#     ax.set_title(f'{title_suffix}\nFN={fn}  FP={fp}  Total cost=${cost:,}',
#                  fontweight='bold', color='#1F3864', fontsize=10)
# plt.tight_layout()
# plt.show()

print('Exercise 5.4 — fill in the threshold values above, then uncomment and run.')
print('See the Answer Key at the end of this chapter.')

---
# 5.5 Pipeline: Saving and Deploying the Defect Detector

## ModelPipeline with image transforms

The `ModelPipeline` class from Chapter 3 (used also in Chapter 4) is extended here for image data. The key difference is what the **preprocessor** slot holds:

- Chapters 3 and 4: the preprocessor was a fitted `StandardScaler`
- Chapter 5: the preprocessor is the `torchvision.transforms` pipeline — the exact sequence of resize, grayscale, normalise operations used during training

Saving the transforms alongside the model guarantees that new images are processed identically to training images before being scored. If the transforms were not saved, a user loading the pipeline would need to know the correct resize dimensions, normalisation statistics, and channel settings — and getting any of these wrong silently corrupts every prediction.

**Input validation for images** checks:
- That the input is a PIL image or numpy array (not a raw file path)
- That after transform, the tensor has the correct shape: `(1, H, W)` for grayscale
- That the expected spatial dimensions match (`H == W == DEFECT_SIZE`)

All other methods — `save`, `load`, `retrain` — are identical to Chapters 3 and 4.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ModelPipeline — image classifier version
#
# Same class as Chapters 3 and 4. Extensions for image data:
#   - preprocessor holds torchvision.transforms (not StandardScaler)
#   - validate_input() checks image shape and channels
#   - predict() applies transforms and returns label + probability
# ─────────────────────────────────────────────────────────────────────────────

import dill

class ModelPipeline:
    """Bundles a trained PyTorch model with its preprocessor and metadata.
    Provides: save | load | validate_input | predict | retrain.
    Image-classifier version: preprocessor holds torchvision.transforms.
    All other methods identical to Chapters 3 and 4.
    """

    def __init__(self, model, model_config, preprocessor=None,
                 feature_names=None, feature_ranges=None, model_class=None):
        self.model            = model
        self.model_config     = model_config
        self.preprocessor     = preprocessor    # torchvision.transforms.Compose
        self.feature_names    = feature_names or []   # not used for images
        self.feature_ranges   = feature_ranges or {}  # holds expected image shape
        self._model_class     = model_class
        self.model_class      = model_class.__name__ if model_class else type(model).__name__
        self.training_history = {'train_loss': [], 'val_loss': []}

    def save(self, path):
        """Save the complete pipeline. dill serialises both model class and transforms."""
        cls_obj     = self._model_class if self._model_class is not None else type(self.model)
        class_bytes = dill.dumps(cls_obj)
        checkpoint  = {
            'model_class'       : self.model_class,
            'model_class_bytes' : class_bytes,
            'model_config'      : self.model_config,
            'state_dict'        : self.model.state_dict(),
            'preprocessor'      : self.preprocessor,   # transforms saved here
            'feature_names'     : self.feature_names,
            'feature_ranges'    : self.feature_ranges,  # expected image shape
            'training_history'  : self.training_history,
            'pytorch_version'   : torch.__version__,
            'saved_at'          : datetime.datetime.now().isoformat(),
        }
        torch.save(checkpoint, path)
        print(f'Pipeline saved -> {path}')
        print(f'  model_class : {self.model_class}')
        print(f'  history     : {len(self.training_history["val_loss"])} epochs recorded')
        print(f'  saved_at    : {checkpoint["saved_at"]}')

    @classmethod
    def load(cls, path, device=None):
        """Load a saved pipeline. No training code needed."""
        device     = device or torch.device('cpu')
        checkpoint = torch.load(path, map_location=device, weights_only=False)
        model_class = dill.loads(checkpoint['model_class_bytes'])
        # Filter config to only keys the model constructor accepts
        cnn_keys = {'num_classes', 'dropout_p'}
        cnn_cfg  = {k: v for k, v in checkpoint['model_config'].items() if k in cnn_keys}
        model    = model_class(**cnn_cfg)
        model.load_state_dict(checkpoint['state_dict'])
        model.to(device)
        model.eval()
        pipeline = cls(
            model          = model,
            model_config   = checkpoint['model_config'],
            preprocessor   = checkpoint.get('preprocessor'),
            feature_names  = checkpoint.get('feature_names', []),
            feature_ranges = checkpoint.get('feature_ranges', {}),
            model_class    = model_class,
        )
        pipeline.training_history = checkpoint.get(
            'training_history', {'train_loss': [], 'val_loss': []})
        print(f'Pipeline loaded <- {path}')
        print(f'  model_class    : {checkpoint.get("model_class")}  (reconstructed from dill)')
        print(f'  history        : {len(pipeline.training_history["val_loss"])} epochs recorded')
        print(f'  saved_at       : {checkpoint.get("saved_at")}')
        print(f'  pytorch_version: {checkpoint.get("pytorch_version")}')
        return pipeline

    def validate_input(self, image):
        """Validate that a PIL image or numpy array is compatible with this pipeline.

        Checks:
          1. Input is a PIL Image or numpy array (not a file path)
          2. After applying transforms, tensor shape matches expected (C, H, W)
        Raises ValueError with a clear message on failure.
        """
        if isinstance(image, (str, pathlib.Path)):
            raise ValueError(
                'Expected a PIL Image or numpy array, not a file path. '
                'Load the image first: Image.open(path)')

        if not isinstance(image, (Image.Image, np.ndarray)):
            raise ValueError(
                f'Expected PIL Image or numpy array, got {type(image).__name__}')

        if self.preprocessor is not None:
            pil_img = Image.fromarray(image) if isinstance(image, np.ndarray) else image
            tensor  = self.preprocessor(pil_img)
            expected_shape = self.feature_ranges.get('expected_tensor_shape')
            if expected_shape and tuple(tensor.shape) != tuple(expected_shape):
                raise ValueError(
                    f'After transforms, expected tensor shape {expected_shape}, '
                    f'got {tuple(tensor.shape)}. '
                    f'Check image channels and size.')
        return True

    def predict(self, images, device=None, threshold=0.5):
        """Predict class for a list of PIL images or numpy arrays.

        Args:
            images    : list of PIL Images or numpy arrays
            device    : torch.device — defaults to CPU
            threshold : P(Defective) above which an image is flagged as Defective

        Returns:
            DataFrame with columns: defect_probability, predicted_label
        """
        device = device or torch.device('cpu')
        if not isinstance(images, list):
            images = [images]

        tensors = []
        for img in images:
            self.validate_input(img)
            pil_img = Image.fromarray(img) if isinstance(img, np.ndarray) else img
            tensors.append(self.preprocessor(pil_img))

        batch = torch.stack(tensors).to(device)  # (N, C, H, W)
        self.model.eval()
        with torch.no_grad():
            probs = torch.softmax(self.model(batch), dim=1)[:, 0].cpu().numpy()  # P(Defective)

        class_labels = self.model_config.get('class_labels', {0: 'Defective', 1: 'OK'})
        return pd.DataFrame({
            'defect_probability': np.round(probs, 4),
            'predicted_label'   : [class_labels[int(p >= threshold)] for p in probs],
        })

    def retrain(self, train_loader, val_loader, criterion, optimiser,
                num_epochs, device=None, patience=None):
        """Continue training. Appends to training_history. Identical to Chapters 3 and 4."""
        device = device or torch.device('cpu')
        es     = EarlyStopping(patience=patience) if patience else None
        for epoch in range(num_epochs):
            tl = train_one_epoch(self.model, train_loader, criterion, optimiser, device)
            vl = evaluate(self.model, val_loader, criterion, device)
            self.training_history['train_loss'].append(tl)
            self.training_history['val_loss'].append(vl)
            if epoch % 5 == 0:
                print(f'  Epoch {epoch:3d}: train={tl:.4f}  val={vl:.4f}')
            if es and es.step(vl, self.model):
                print(f'  Early stopping at epoch {epoch}  (best val: {es.best_loss:.4f})')
                es.restore_best(self.model)
                break


print('ModelPipeline defined.')
print('Methods: save | load | validate_input | predict | retrain')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Wrap the defect model in ModelPipeline and save
#
# feature_ranges stores the expected tensor shape after transforms —
# used by validate_input() to catch images of the wrong size or channel count.
# ─────────────────────────────────────────────────────────────────────────────

import pathlib

# Compute the expected tensor shape from the transform pipeline
dummy_pil = Image.new('L', (DEFECT_SIZE, DEFECT_SIZE))
expected_shape = tuple(defect_val_transform(dummy_pil).shape)   # (1, 64, 64)

defect_model_config = {
    **defect_config,
    'class_labels': {0: 'Defective', 1: 'OK'},
}

pipeline = ModelPipeline(
    model          = defect_model,
    model_config   = defect_model_config,
    preprocessor   = defect_val_transform,   # inference transform (no augmentation)
    feature_ranges = {'expected_tensor_shape': expected_shape},
    model_class    = DefectCNN,
)
pipeline.training_history = {'train_loss': defect_train_hist, 'val_loss': defect_val_hist}

pipeline.save('defect_model_v1.pth')
print(f'\nExpected input shape after transforms: {expected_shape}')
print('The transforms are saved inside the .pth file — no external config needed.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load in a clean context and score 5 new product images
# ─────────────────────────────────────────────────────────────────────────────

del pipeline   # simulate a fresh session

loaded = ModelPipeline.load('defect_model_v1.pth')
print(f'\nTraining history restored: {len(loaded.training_history["val_loss"])} epochs')

# Load 5 test images as PIL Images — 3 OK, 2 defective
ok_paths  = get_images(data_root / 'test' / 'ok_front',  n=3)
def_paths = get_images(data_root / 'test' / 'def_front', n=2)
test_images  = [Image.open(p) for p in ok_paths + def_paths]
true_labels  = ['OK'] * 3 + ['Defective'] * 2

results = loaded.predict(test_images, threshold=best_t)
results['true_label'] = true_labels

print(f'\nPredictions (threshold = {best_t:.2f}):')
print(results.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Input validation — three failure cases
# ─────────────────────────────────────────────────────────────────────────────

print('Case 1 — file path passed instead of PIL Image:')
try:
    loaded.predict(str(ok_paths[0]))
except (ValueError, TypeError) as e:
    print(f'  Caught: {e}')

print()
print('Case 2 — wrong data type (integer):')
try:
    loaded.validate_input(42)
except ValueError as e:
    print(f'  Caught: {e}')

print()
print('Case 3 — image wrong size (results in wrong tensor shape after transform):')
# Create a tiny 10×10 image — after resize to 64×64 it would be fine,
# but passing a different expected_shape lets us trigger the error
wrong_pipeline = ModelPipeline(
    model          = loaded.model,
    model_config   = loaded.model_config,
    preprocessor   = loaded.preprocessor,
    feature_ranges = {'expected_tensor_shape': (1, 32, 32)},  # deliberately wrong
    model_class    = None,
)
try:
    wrong_pipeline.validate_input(test_images[0])
except ValueError as e:
    print(f'  Caught: {e}')

print()
print('Case 4 — valid input (all checks pass):')
print(f'  validate_input returned: {loaded.validate_input(test_images[0])}  (True)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Retrain on a new batch of inspected images
#
# Scenario: the production line has run another week and the quality team
# has labelled 500 new images. We update the model.
#
# retrain() signature identical to Chapters 3 and 4:
#   retrain(train_loader, val_loader, criterion, optimiser, num_epochs, device)
# ─────────────────────────────────────────────────────────────────────────────

retrain_opt = optim.SGD(loaded.model.parameters(), lr=1e-3, momentum=0.9)

print('Retraining on new batch (10 additional epochs):')
loaded.retrain(
    train_loader = defect_train_loader,
    val_loader   = defect_val_loader,
    criterion    = nn.CrossEntropyLoss(),
    optimiser    = retrain_opt,
    num_epochs   = 10,
    device       = device,
)

loaded.save('defect_model_v2.pth')
print(f'\nTotal epochs in history: {len(loaded.training_history["val_loss"])}')
print('Version 1 preserved for rollback if needed.')

---
## Answer Key — Chapter 5 Exercises

---

### Exercise 5.1 — Apply a Sharpening Kernel

```python
kernel_sharp = torch.tensor([[
    [ 0., -1.,  0.],
    [-1.,  5., -1.],
    [ 0., -1.,  0.]
]]).unsqueeze(0)   # shape: (1, 1, 3, 3)

with torch.no_grad():
    out_sharp = F.conv2d(img_4d, kernel_sharp, padding=1).squeeze()

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
fig.patch.set_facecolor('white')
axes[0].imshow(img_tensor.squeeze(), cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(out_sharp.numpy(), cmap='gray');   axes[1].set_title('Sharpened'); axes[1].axis('off')
plt.tight_layout()
plt.show()
```

**Why it works:** The centre value (5) amplifies the pixel itself. The four direct neighbours are subtracted (−1 each). This is mathematically equivalent to adding the original image minus a blurred version — enhancing edges. Increasing the centre from 5 to 9 strengthens the effect; values below 5 reduce it. The corners are 0, so diagonal neighbours are ignored.

---

### Exercise 5.2 — Build a Deeper CNN

```python
class DeepFashionCNN(nn.Module):
    def __init__(self, num_classes=10, dropout_p=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,  32,  kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64,  kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256),  # 128 channels × 3 × 3 spatial = 1,152
            nn.ReLU(), nn.Dropout(dropout_p),
            nn.Linear(256, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))
```

**Reasoning:** After each `MaxPool2d(2, 2)`, the spatial size halves: 28 → 14 → 7 → 3 (floor(7/2) = 3). After three blocks, we have 128 feature maps each of size 3×3 = 1,152 values flattened into the FC head. The third Conv2d takes 64 input channels and produces 128, following the channel-doubling convention (32 → 64 → 128).

---

### Exercise 5.3 — Measure the Effect of Augmentation

This exercise has no fixed answer — the result depends on your training run. What to look for:

- **Train/val loss gap** should be larger without augmentation. The model memorises training images more readily when it sees the exact same image every epoch.
- **Final val loss** should be lower (better) with augmentation, because the model generalises more from seeing varied views.
- The improvement is usually modest on Fashion-MNIST (small dataset, low resolution), but clearly visible on larger datasets like the casting images in §5.4.

---

### Exercise 5.4 — Tune the Threshold

```python
threshold_default = 0.5    # standard midpoint — treat P(Defective) ≥ 0.5 as defective
threshold_optimal = best_t  # computed in Figure 5.6 cost analysis
```

**What to observe:** At the optimal threshold (typically lower than 0.5 for this dataset), the confusion matrix will show:
- Fewer False Negatives (bottom-left cell) — fewer defective parts escaping inspection
- More False Positives (top-right cell) — more conforming parts flagged for re-inspection

The total cost at the optimal threshold is lower than at 0.5 because the FN cost ($500) far outweighs the FP cost ($20). Accepting 25 extra false alarms is justified if it prevents even one defective part reaching a customer.

---
## Chapter 5 Summary

| Concept | Key takeaway |
|---------|-------------|
| **Why CNNs for images** | FFNs destroy spatial structure by flattening; CNNs preserve it through local, shared kernels |
| **Convolution** | A small kernel slides across the image; the dot product at each position produces a feature map |
| **Parameter sharing** | The same kernel weights are reused at every position — CNNs are far more parameter-efficient than FFNs on images |
| **Stride and padding** | Stride controls down-sampling; `padding=1` on a 3×3 kernel keeps spatial size constant |
| **Max pooling** | Halves spatial dimensions, reduces computation, adds slight translation invariance |
| **CNN building block** | `Conv2d → ReLU → MaxPool2d` — channels grow, spatial dimensions shrink |
| **SGD + Momentum** | Recommended optimiser for CNNs (Chapter 3, Table 3.2) — accumulates velocity, reduces ravine oscillation |
| **Data augmentation** | Random flips, rotations, and shifts at train time act as a regulariser — only applied during training |
| **Learned filters** | Kernels are discovered by gradient descent — many resemble edge detectors without being programmed to |
| **Threshold tuning** | In manufacturing, FN >> FP cost → optimal threshold is lower than 0.5 to catch more defects |
| **ModelPipeline** | Preprocessor slot holds `torchvision.transforms`; `validate_input()` checks image shape and channels |

---

## What Comes Next

Chapter 6 introduces **Recurrent Neural Networks and LSTMs** — the architecture built for sequential data. Where a CNN processes all spatial positions simultaneously, an RNN processes data one step at a time, maintaining a hidden state that carries information forward through the sequence. The two business applications are demand forecasting (predicting future sales from historical records) and sentiment analysis (classifying customer reviews as positive or negative).

---
*Deep Learning for Business Analytics: From Basics to Large Language Models*  
*Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter*